# Cluster ID to Tool Mapping Inspector

This notebook helps you inspect cluster IDs and their associated tool names from the
cluster retrieval checkpoint.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(project_root))

project_root


PosixPath('/scratch4/home/akrik/NTILC')

In [2]:
import torch
import pandas as pd

from models.software_layer import ClusterToolMapper


/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
RETRIEVAL_CKPT = project_root / 'checkpoints/cluster_retrieval/best_model.pt'

if not RETRIEVAL_CKPT.exists():
    raise FileNotFoundError(f'Checkpoint not found: {RETRIEVAL_CKPT}')

RETRIEVAL_CKPT


PosixPath('/scratch4/home/akrik/NTILC/checkpoints/cluster_retrieval/best_model.pt')

In [4]:
checkpoint = torch.load(RETRIEVAL_CKPT, map_location='cpu', weights_only=False)

tool_names = checkpoint.get('tool_names', [])
cluster_centroids = checkpoint.get('cluster_centroids')
tool_to_idx = checkpoint.get('tool_to_idx', {})

print('Checkpoint keys:', sorted(checkpoint.keys()))
print('Num tool_names:', len(tool_names))
print('Centroid shape:', tuple(cluster_centroids.shape) if cluster_centroids is not None else None)
print('Num tool_to_idx entries:', len(tool_to_idx) if isinstance(tool_to_idx, dict) else 0)


Checkpoint keys: ['cluster_centroids', 'config', 'epoch', 'optimizer_state_dict', 'phase1_checkpoint', 'query_encoder_state_dict', 'tool_names', 'tool_to_idx', 'val_metrics']
Num tool_names: 1016
Centroid shape: (1016, 128)
Num tool_to_idx entries: 1016


In [5]:
mapper = ClusterToolMapper.from_retrieval_checkpoint(RETRIEVAL_CKPT)
mapping_dict = mapper.to_dict()['cluster_to_tool']

mapping_df = pd.DataFrame(
    [{'cluster_id': int(cid), 'tool': tool} for cid, tool in mapping_dict.items()]
).sort_values('cluster_id').reset_index(drop=True)

mapping_df


,cluster_id,tool
0,0,PCPCompat
1,1,PCPIntro
2,2,abicompat
3,3,abidb
4,4,abilint
...,...,...
1011,1011,yum-groups-manager
1012,1012,yum-utils
1013,1013,yum-versionlock
1014,1014,yumdownloader


In [6]:
print('Num clusters:', len(mapping_df))
print('Unique tools:', mapping_df['tool'].nunique())
dupe_tools = mapping_df[mapping_df['tool'].duplicated(keep=False)].sort_values('tool')
print('Duplicate tool mappings:', len(dupe_tools))
dupe_tools.head(20)


Num clusters: 1016
Unique tools: 1016
Duplicate tool mappings: 0


,cluster_id,tool


## Lookup Helpers
Use the two helper functions below to inspect a specific cluster or search by tool substring.


In [7]:
def inspect_cluster(cluster_id: int):
    row = mapping_df[mapping_df['cluster_id'] == int(cluster_id)]
    if row.empty:
        print(f'Cluster {cluster_id} not found.')
        return
    display(row)

def find_tool(text: str):
    text = str(text).strip().lower()
    if not text:
        return mapping_df
    out = mapping_df[mapping_df['tool'].str.lower().str.contains(text, regex=False)]
    return out.sort_values('cluster_id').reset_index(drop=True)


In [8]:
# Example lookups
inspect_cluster(0)
find_tool('grep').head(20)


,cluster_id,tool
0,0,PCPCompat


,cluster_id,tool
0,239,git-bugreport
1,522,msggrep


## Optional Export
Export the mapping to CSV if needed.


In [ ]:
# out_csv = project_root / 'output/cluster_to_tool_mapping.csv'
# out_csv.parent.mkdir(parents=True, exist_ok=True)
# mapping_df.to_csv(out_csv, index=False)
# out_csv
